# 🏀 Ensemble Predictions with REAL LightGBM Model

**Strategy**: Combine trained LightGBM model with team strength baseline
- **LightGBM** (64.5% test accuracy): Trained on 1722 games with quantile regression
- **Team Strength** (53% accuracy): Simple but interpretable
- **Ensemble** (60-65% target): Best of both worlds

**Weights**: 70% LightGBM + 30% Team Strength
**Model Performance**: 64.5% accuracy on chronological test set

In [43]:
import sys
sys.path.insert(0, r'c:\Users\Windows User\My_folder\gamble_code\sports_analytics')

import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful")
print(f"📅 Prediction date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ All imports successful
📅 Prediction date: 2026-02-16 22:10:25


In [44]:
# ============================================================
# LOAD TRAINED LIGHTGBM MODEL
# ============================================================
print("\n" + "="*80)
print("🤖 LOADING TRAINED LIGHTGBM MODEL")
print("="*80)

from train_lgbm_model import LGBMWinPredictor
from machine_learning.data_loader import (
    fetch_nba_games, 
    calculate_rolling_stats,
    create_matchup_features,
    get_all_nba_teams
)

# Load trained model
print("\n📂 Loading model from disk...")
lgbm_model = LGBMWinPredictor.load('machine_learning/models/lgbm_win_predictor_latest.pkl')
print(f"✅ Model loaded successfully")
print(f"   Features: {len(lgbm_model.feature_names)}")

# Load latest data for rolling stats
print("\n📥 Loading NBA data for feature calculation...")
games_df = fetch_nba_games(seasons=['2024-25'], season_type='Regular Season', verbose=False)
games_with_stats = calculate_rolling_stats(games_df, window=5)
print(f"✅ Loaded {len(games_with_stats)} games with rolling stats")
print(f"   Date range: {games_with_stats['GAME_DATE'].min()} → {games_with_stats['GAME_DATE'].max()}")

# Get team mappings
team_data = get_all_nba_teams()
team_name_to_id = {team['full_name']: team['id'] for team in team_data['teams']}

print("\n✅ Data ready for predictions")


🤖 LOADING TRAINED LIGHTGBM MODEL

📂 Loading model from disk...
📂 Model loaded from machine_learning/models/lgbm_win_predictor_latest.pkl
✅ Model loaded successfully
   Features: 24

📥 Loading NBA data for feature calculation...
✅ Loaded 2460 games with rolling stats
   Date range: 2024-10-22 00:00:00 → 2025-04-13 00:00:00

✅ Data ready for predictions


In [45]:
# ============================================================
# METHOD 1: TEAM STRENGTH PREDICTIONS (Baseline)
# ============================================================
print("\n" + "="*80)
print("🏀 METHOD 1: TEAM STRENGTH RATINGS (BASELINE)")
print("="*80)

import math

# Team strength ratings (empirical from recent performance)
TEAM_STRENGTH = {
    'Philadelphia 76ers': 92,
    'Atlanta Hawks': 81,
    'Indiana Pacers': 88,
    'Washington Wizards': 79,
    'New York Knicks': 90,
    'Detroit Pistons': 67,
    'Chicago Bulls': 84,
    'Toronto Raptors': 78,
    'Phoenix Suns': 91,
    'San Antonio Spurs': 77,
    'Boston Celtics': 93,
    'Golden State Warriors': 86,
    'Sacramento Kings': 83,
    'Orlando Magic': 76,
    'Denver Nuggets': 92,
    'Los Angeles Clippers': 87
}

def predict_team_strength(away_team, home_team):
    """Predict using team strength ratings"""
    away_strength = TEAM_STRENGTH.get(away_team, 80)
    home_strength = TEAM_STRENGTH.get(home_team, 80)
    
    # Home court advantage: ~2.5 points
    home_adjusted = home_strength + 2.5
    
    # Strength differential
    diff = home_adjusted - away_strength
    
    # Convert to probability using logistic function
    home_prob = 1 / (1 + math.exp(-diff / 10))
    
    if home_prob > 0.5:
        winner = home_team
        prob = home_prob
        spread = diff * 0.8
    else:
        winner = away_team
        prob = 1 - home_prob
        spread = -diff * 0.8
    
    # Confidence based on probability margin
    prob_margin = abs(home_prob - 0.5) * 2
    if prob_margin >= 0.50:
        confidence = 'HIGH'
    elif prob_margin >= 0.30:
        confidence = 'MEDIUM'
    else:
        confidence = 'LOW'
    
    return {
        'winner': winner,
        'spread': spread,
        'probability': prob if winner == home_team else 1 - prob,
        'confidence': confidence,
        'confidence_score': prob_margin
    }

print("✅ Team strength model initialized")
print(f"   Teams available: {len(TEAM_STRENGTH)}")


🏀 METHOD 1: TEAM STRENGTH RATINGS (BASELINE)
✅ Team strength model initialized
   Teams available: 16


In [46]:
# ============================================================
# METHOD 2: REAL LIGHTGBM PREDICTIONS
# ============================================================
print("\n" + "="*80)
print("🤖 METHOD 2: TRAINED LIGHTGBM MODEL")
print("="*80)

def get_team_latest_features(team_name, games_df_with_stats):
    """
    Get the most recent rolling stats for a team.
    
    Args:
        team_name: Full team name (e.g., 'Philadelphia 76ers')
        games_df_with_stats: DataFrame with rolling stats
    
    Returns:
        dict of feature_name: value
    """
    team_id = team_name_to_id.get(team_name)
    if not team_id:
        return {}
    
    # Get team's games, sorted by date
    team_games = games_df_with_stats[games_df_with_stats['TEAM_ID'] == team_id].sort_values('GAME_DATE')
    
    if len(team_games) == 0:
        return {}
    
    # Get most recent game's stats
    latest = team_games.iloc[-1]
    
    # Extract rolling features
    features = {}
    for col in games_df_with_stats.columns:
        if '_ROLL' in col or col in ['WIN_STREAK', 'REST_DAYS', 'IS_BACK_TO_BACK', 'WIN_RATE_10']:
            features[col] = latest[col] if pd.notna(latest[col]) else 0.0
    
    return features

def predict_lgbm_real(away_team, home_team):
    """
    Predict using trained LightGBM model.
    
    Args:
        away_team: Away team name
        home_team: Home team name
    
    Returns:
        dict with winner, spread, probability, confidence
    """
    # Get latest features for both teams
    home_features = get_team_latest_features(home_team, games_with_stats)
    away_features = get_team_latest_features(away_team, games_with_stats)
    
    if not home_features or not away_features:
        # Fallback to team strength if features not available
        return predict_team_strength(away_team, home_team)
    
    # Create matchup feature vector in expected order
    feature_row = {}
    for col, value in home_features.items():
        feature_row[f'HOME_{col}'] = value
    for col, value in away_features.items():
        feature_row[f'AWAY_{col}'] = value
    
    # Convert to DataFrame with correct feature order
    X = pd.DataFrame([feature_row])
    
    # Align with model's feature names (fill missing with 0)
    for feat in lgbm_model.feature_names:
        if feat not in X.columns:
            X[feat] = 0.0
    X = X[lgbm_model.feature_names]
    
    # Predict with model
    preds = lgbm_model.predict_win_probability(X)
    
    # Extract predictions
    home_win_prob = preds['win_prob'][0]
    point_diff = preds['point_diff'][0]
    uncertainty = preds['uncertainty'][0]
    
    # Determine winner
    if home_win_prob > 0.5:
        winner = home_team
        prob = home_win_prob
        spread = point_diff
    else:
        winner = away_team
        prob = 1 - home_win_prob
        spread = -point_diff
    
    # Confidence from probability margin
    prob_margin = abs(home_win_prob - 0.5) * 2
    if prob_margin >= 0.50:
        confidence = 'HIGH'
    elif prob_margin >= 0.30:
        confidence = 'MEDIUM'
    else:
        confidence = 'LOW'
    
    return {
        'winner': winner,
        'spread': spread,
        'probability': prob,
        'confidence': confidence,
        'confidence_score': prob_margin,
        'uncertainty': uncertainty / 10.0  # Normalize to 0-1 range
    }

print("✅ LightGBM predictor function ready")
print(f"   Model features: {len(lgbm_model.feature_names)}")


🤖 METHOD 2: TRAINED LIGHTGBM MODEL
✅ LightGBM predictor function ready
   Model features: 24


In [47]:
# ============================================================
# METHOD 3: ENSEMBLE PREDICTIONS
# ============================================================
print("\n" + "="*80)
print("🎯 METHOD 3: ENSEMBLE (70% LightGBM + 30% Team Strength)")
print("="*80)

# Weights from validation testing
ENSEMBLE_WEIGHTS = {
    'lgbm': 0.70,           # LightGBM more accurate (64.5% test)
    'team_strength': 0.30   # Team strength as baseline (53%)
}

def predict_ensemble(away_team, home_team):
    """Combine LightGBM and team strength predictions"""
    
    # Get both predictions
    ts_pred = predict_team_strength(away_team, home_team)
    lgbm_pred = predict_lgbm_real(away_team, home_team)
    
    # Ensemble weights
    lgbm_weight = ENSEMBLE_WEIGHTS['lgbm']
    ts_weight = ENSEMBLE_WEIGHTS['team_strength']
    
    # Blend spreads
    ensemble_spread = (lgbm_pred['spread'] * lgbm_weight + 
                      ts_pred['spread'] * ts_weight)
    
    # Blend probabilities
    if lgbm_pred['winner'] == ts_pred['winner']:
        # Both agree - high confidence
        ensemble_prob = (lgbm_pred['probability'] * lgbm_weight + 
                        ts_pred['probability'] * ts_weight)
        ensemble_winner = lgbm_pred['winner']
        confidence_boost = 0.15
    else:
        # Disagree - use weighted probability, lower confidence
        lgbm_prob = lgbm_pred['probability'] if lgbm_pred['winner'] == away_team else 1 - lgbm_pred['probability']
        ts_prob = ts_pred['probability'] if ts_pred['winner'] == away_team else 1 - ts_pred['probability']
        
        away_prob = lgbm_prob * lgbm_weight + ts_prob * ts_weight
        ensemble_prob = away_prob if away_prob > 0.5 else 1 - away_prob
        ensemble_winner = away_team if away_prob > 0.5 else ts_pred['winner']
        confidence_boost = -0.10
    
    # Confidence from agreement
    avg_confidence = (lgbm_pred['confidence_score'] + ts_pred['confidence_score']) / 2
    adjusted_confidence = avg_confidence + confidence_boost
    
    if adjusted_confidence >= 0.50:
        confidence = 'HIGH'
    elif adjusted_confidence >= 0.30:
        confidence = 'MEDIUM'
    else:
        confidence = 'LOW'
    
    # Prediction interval (uncertainty)
    avg_uncertainty = (lgbm_pred.get('uncertainty', 0.3) + 0.3) / 2
    
    return {
        'winner': ensemble_winner,
        'spread': ensemble_spread,
        'probability': ensemble_prob,
        'confidence': confidence,
        'confidence_score': adjusted_confidence,
        'uncertainty': avg_uncertainty,
        'methods_agree': lgbm_pred['winner'] == ts_pred['winner']
    }

print("✅ Ensemble model initialized")
print(f"   Weights: {ENSEMBLE_WEIGHTS['lgbm']:.0%} LightGBM + {ENSEMBLE_WEIGHTS['team_strength']:.0%} Team Strength")


🎯 METHOD 3: ENSEMBLE (70% LightGBM + 30% Team Strength)
✅ Ensemble model initialized
   Weights: 70% LightGBM + 30% Team Strength


In [48]:
# ============================================================
# FEBRUARY 19, 2026 GAMES
# ============================================================
print("\n" + "="*80)
print("📅 GENERATING PREDICTIONS FOR FEBRUARY 19, 2026")
print("="*80)

# Games for February 19, 2026
GAMES_FEB19 = [
    {'away': 'Atlanta Hawks', 'home': 'Philadelphia 76ers', 'time': '7:00p'},
    {'away': 'Indiana Pacers', 'home': 'Washington Wizards', 'time': '7:00p'},
    {'away': 'Detroit Pistons', 'home': 'New York Knicks', 'time': '7:30p'},
    {'away': 'Toronto Raptors', 'home': 'Chicago Bulls', 'time': '8:00p'},
    {'away': 'Phoenix Suns', 'home': 'San Antonio Spurs', 'time': '8:30p'},
    {'away': 'Boston Celtics', 'home': 'Golden State Warriors', 'time': '10:00p'},
    {'away': 'Orlando Magic', 'home': 'Sacramento Kings', 'time': '10:00p'},
    {'away': 'Denver Nuggets', 'home': 'Los Angeles Clippers', 'time': '10:30p'},
]

# Generate ensemble predictions
print(f"\nGenerating ensemble predictions for {len(GAMES_FEB19)} games...\n")

all_predictions = []
for i, game in enumerate(GAMES_FEB19, 1):
    print(f"   Predicting Game {i}/{len(GAMES_FEB19)}: {game['away']} @ {game['home']}...")
    pred = predict_ensemble(game['away'], game['home'])
    pred['game_num'] = i
    pred['away_team'] = game['away']
    pred['home_team'] = game['home']
    pred['time'] = game['time']
    all_predictions.append(pred)

# Create DataFrame for analysis
predictions_df = pd.DataFrame(all_predictions)

print(f"\n✅ Predictions generated for all {len(GAMES_FEB19)} games")
print(f"\nAccuracy expectations (based on backtest):")
print(f"   • LightGBM test accuracy: 64.5%")
print(f"   • Team strength: ~53%")
print(f"   • Expected ensemble: ~60-65%")
print(f"   • HIGH confidence games: ~75-82% accuracy")
print(f"   • MEDIUM confidence games: ~62-69% accuracy")
print(f"   • LOW confidence games: ~50-56% accuracy")


📅 GENERATING PREDICTIONS FOR FEBRUARY 19, 2026

Generating ensemble predictions for 8 games...

   Predicting Game 1/8: Atlanta Hawks @ Philadelphia 76ers...
   Predicting Game 2/8: Indiana Pacers @ Washington Wizards...
   Predicting Game 3/8: Detroit Pistons @ New York Knicks...
   Predicting Game 4/8: Toronto Raptors @ Chicago Bulls...
   Predicting Game 5/8: Phoenix Suns @ San Antonio Spurs...
   Predicting Game 6/8: Boston Celtics @ Golden State Warriors...
   Predicting Game 7/8: Orlando Magic @ Sacramento Kings...
   Predicting Game 8/8: Denver Nuggets @ Los Angeles Clippers...

✅ Predictions generated for all 8 games

Accuracy expectations (based on backtest):
   • LightGBM test accuracy: 64.5%
   • Team strength: ~53%
   • Expected ensemble: ~60-65%
   • HIGH confidence games: ~75-82% accuracy
   • MEDIUM confidence games: ~62-69% accuracy
   • LOW confidence games: ~50-56% accuracy


In [49]:
# ============================================================
# DETAILED PREDICTION DISPLAY
# ============================================================
print("\n" + "="*80)
print("🎯 DETAILED PREDICTIONS FOR FEBRUARY 19, 2026")
print("="*80)

for pred in all_predictions:
    spread_sign = "+" if pred['spread'] > 0 else ""
    
    print(f"\n{'─'*80}")
    print(f"GAME {pred['game_num']}: {pred['away_team']:20} @ {pred['home_team']:20} ({pred['time']})")
    print(f"{'─'*80}")
    print(f"  🏆 Predicted Winner:    {pred['winner']}")
    print(f"  📊 Spread:              {spread_sign}{pred['spread']:.1f} points")
    print(f"  🎯 Win Probability:     {pred['probability']:.1%}")
    print(f"  💪 Confidence Level:    {pred['confidence']}")
    print(f"  📈 Confidence Score:    {pred['confidence_score']:.3f}")
    
    if pred['methods_agree']:
        print(f"  ✅ Both methods agree (higher confidence)")
    else:
        print(f"  ⚠️  Methods disagree (lower confidence)")

print(f"\n{'─'*80}\n")


🎯 DETAILED PREDICTIONS FOR FEBRUARY 19, 2026

────────────────────────────────────────────────────────────────────────────────
GAME 1: Atlanta Hawks        @ Philadelphia 76ers   (7:00p)
────────────────────────────────────────────────────────────────────────────────
  🏆 Predicted Winner:    Philadelphia 76ers
  📊 Spread:              +6.3 points
  🎯 Win Probability:     53.3%
  💪 Confidence Level:    LOW
  📈 Confidence Score:    0.273
  ⚠️  Methods disagree (lower confidence)

────────────────────────────────────────────────────────────────────────────────
GAME 2: Indiana Pacers       @ Washington Wizards   (7:00p)
────────────────────────────────────────────────────────────────────────────────
  🏆 Predicted Winner:    Indiana Pacers
  📊 Spread:              +9.4 points
  🎯 Win Probability:     58.6%
  💪 Confidence Level:    MEDIUM
  📈 Confidence Score:    0.497
  ✅ Both methods agree (higher confidence)

───────────────────────────────────────────────────────────────────────────────

In [50]:
# ============================================================
# SUMMARY & STATISTICS
# ============================================================
print("\n" + "="*80)
print("📊 PREDICTION SUMMARY")
print("="*80)

# Count confidence levels
high_conf = (predictions_df['confidence'] == 'HIGH').sum()
med_conf = (predictions_df['confidence'] == 'MEDIUM').sum()
low_conf = (predictions_df['confidence'] == 'LOW').sum()

# Count method agreement
agree = predictions_df['methods_agree'].sum()
disagree = len(predictions_df) - agree

print(f"\n📈 Confidence Distribution:")
print(f"   HIGH Confidence:        {high_conf} games")
print(f"   MEDIUM Confidence:      {med_conf} games")
print(f"   LOW Confidence:         {low_conf} games")

print(f"\n🎯 Method Agreement:")
print(f"   Both methods agree:     {agree} games")
print(f"   Methods disagree:       {disagree} games")

print(f"\n💻 Predicted Winners:")
for i, pred in enumerate(all_predictions, 1):
    print(f"   Game {i}: {pred['winner']:20} ({pred['confidence']})")

# Spread analysis
print(f"\n📊 Spread Analysis:")
avg_spread = abs(predictions_df['spread']).mean()
max_spread = abs(predictions_df['spread']).max()
min_spread = abs(predictions_df['spread']).min()

print(f"   Average spread:         {avg_spread:.1f} points")
print(f"   Max spread:             {max_spread:.1f} points ({predictions_df.loc[predictions_df['spread'].abs().idxmax(), 'winner']} favored)")
print(f"   Min spread:             {min_spread:.1f} points (closest game)")

# Probability analysis
avg_prob = predictions_df['probability'].mean()
print(f"\n🎲 Probability Analysis:")
print(f"   Average win prob:       {avg_prob:.1%}")
print(f"   Range:                  {predictions_df['probability'].min():.1%} to {predictions_df['probability'].max():.1%}")

print(f"\n" + "="*80)
print("✅ ENSEMBLE PREDICTION COMPLETE (REAL LIGHTGBM)")
print("="*80 + "\n")


📊 PREDICTION SUMMARY

📈 Confidence Distribution:
   HIGH Confidence:        1 games
   MEDIUM Confidence:      3 games
   LOW Confidence:         4 games

🎯 Method Agreement:
   Both methods agree:     4 games
   Methods disagree:       4 games

💻 Predicted Winners:
   Game 1: Philadelphia 76ers   (LOW)
   Game 2: Indiana Pacers       (MEDIUM)
   Game 3: New York Knicks      (HIGH)
   Game 4: Chicago Bulls        (MEDIUM)
   Game 5: Phoenix Suns         (LOW)
   Game 6: Boston Celtics       (MEDIUM)
   Game 7: Sacramento Kings     (LOW)
   Game 8: Denver Nuggets       (LOW)

📊 Spread Analysis:
   Average spread:         5.0 points
   Max spread:             9.4 points (Indiana Pacers favored)
   Min spread:             2.2 points (closest game)

🎲 Probability Analysis:
   Average win prob:       56.5%
   Range:                  50.4% to 66.2%

✅ ENSEMBLE PREDICTION COMPLETE (REAL LIGHTGBM)



In [51]:
# ============================================================
# MONTE CARLO SIMULATIONS (10,000 PER GAME)
# ============================================================
print("\n" + "="*80)
print("🎲 MONTE CARLO SIMULATIONS")
print("="*80)
print("Running 10,000 simulations per game using quantile-based triangular distributions...\n")

import numpy as np
from scipy import stats as scipy_stats

def triangular_sample(q10, q50, q90, n_samples=1):
    """Sample from asymmetric triangular distribution using three quantiles."""
    left = q10
    right = q90
    mode = q50
    
    if right <= left:
        return np.full(n_samples, mode)
    
    c = (mode - left) / (right - left)
    c = np.clip(c, 0.01, 0.99)
    
    samples = scipy_stats.triang.rvs(c, loc=left, scale=right - left, size=n_samples)
    return samples

def get_team_latest_features(team_name, team_name_to_id, games_df_with_stats):
    """Get the most recent rolling stats for a team."""
    team_id = team_name_to_id.get(team_name)
    if not team_id:
        return {}
    
    team_games = games_df_with_stats[games_df_with_stats['TEAM_ID'] == team_id].sort_values('GAME_DATE')
    
    if len(team_games) == 0:
        return {}
    
    latest = team_games.iloc[-1]
    
    features = {}
    for col in games_df_with_stats.columns:
        if '_ROLL' in col or col in ['WIN_STREAK', 'REST_DAYS', 'IS_BACK_TO_BACK', 'WIN_RATE_10']:
            features[col] = latest[col] if pd.notna(latest[col]) else 0.0
    
    return features

def run_monte_carlo_for_game(model, home_team, away_team, team_name_to_id, games_df_with_stats, n_simulations=10000):
    """Run Monte Carlo simulation for a single game."""
    # Get latest features for both teams
    home_features = get_team_latest_features(home_team, team_name_to_id, games_df_with_stats)
    away_features = get_team_latest_features(away_team, team_name_to_id, games_df_with_stats)
    
    if not home_features or not away_features:
        raise ValueError(f"Missing features for {home_team} or {away_team}")
    
    # Create matchup feature vector
    feature_row = {}
    for col, value in home_features.items():
        feature_row[f'HOME_{col}'] = value
    for col, value in away_features.items():
        feature_row[f'AWAY_{col}'] = value
    
    X = pd.DataFrame([feature_row])
    
    # Align with model's feature names
    for feat in model.feature_names:
        if feat not in X.columns:
            X[feat] = 0.0
    X = X[model.feature_names]
    
    # Get quantile predictions
    X_scaled = model.scaler.transform(X)
    quantiles = model.quantile_model.predict(X_scaled)
    
    q10 = quantiles['q10'][0]
    q50 = quantiles['q50'][0]
    q90 = quantiles['q90'][0]
    
    # Run simulations
    point_diffs = triangular_sample(q10, q50, q90, n_simulations)
    home_wins = (point_diffs > 0).astype(int)
    
    # Calculate statistics
    mc_win_prob = home_wins.mean()
    mc_median_spread = np.median(point_diffs)
    mc_mean_spread = np.mean(point_diffs)
    mc_std_spread = np.std(point_diffs)
    
    ci_95_low, ci_95_high = np.percentile(point_diffs, [2.5, 97.5])
    ci_80_low, ci_80_high = np.percentile(point_diffs, [10, 90])
    
    # Model's calibrated prediction for comparison
    model_pred = model.predict_win_probability(X)
    
    return {
        'mc_win_prob_home': mc_win_prob,
        'mc_median_spread': mc_median_spread,
        'mc_mean_spread': mc_mean_spread,
        'mc_std_spread': mc_std_spread,
        'ci_95_low': ci_95_low,
        'ci_95_high': ci_95_high,
        'ci_80_low': ci_80_low,
        'ci_80_high': ci_80_high,
        'q10': q10,
        'q50': q50,
        'q90': q90,
        'model_win_prob': model_pred['win_prob'][0],
        'model_confidence': model_pred['confidence_label'][0],
        'model_confidence_score': model_pred['confidence_score'][0],
    }

# Run Monte Carlo for all games
mc_results = []

for i, game in enumerate(GAMES_FEB19, 1):
    away_team = game['away']
    home_team = game['home']
    
    print(f"🎲 Game {i}/8: {away_team} @ {home_team}...", end='')
    
    try:
        mc_result = run_monte_carlo_for_game(
            lgbm_model, home_team, away_team, team_name_to_id, games_with_stats, n_simulations=10000
        )
        
        # Determine winner
        if mc_result['mc_win_prob_home'] > 0.5:
            winner = home_team
            winner_prob = mc_result['mc_win_prob_home']
        else:
            winner = away_team
            winner_prob = 1 - mc_result['mc_win_prob_home']
        
        mc_results.append({
            'game_num': i,
            'away_team': away_team,
            'home_team': home_team,
            'time': game['time'],
            'mc_winner': winner,
            'mc_win_prob': winner_prob,
            'mc_median_spread': mc_result['mc_median_spread'],
            'mc_mean_spread': mc_result['mc_mean_spread'],
            'mc_std_spread': mc_result['mc_std_spread'],
            'ci_95_low': mc_result['ci_95_low'],
            'ci_95_high': mc_result['ci_95_high'],
            'ci_80_low': mc_result['ci_80_low'],
            'ci_80_high': mc_result['ci_80_high'],
            'q10': mc_result['q10'],
            'q50': mc_result['q50'],
            'q90': mc_result['q90'],
            'model_confidence': mc_result['model_confidence'],
        })
        
        print(f" ✅ {winner} ({winner_prob:.1%})")
        
    except Exception as e:
        print(f" ⚠️ Error: {e}")
        continue

mc_df = pd.DataFrame(mc_results)

print(f"\n✅ Monte Carlo complete: {len(mc_results)} games simulated (80,000 total simulations)")


🎲 MONTE CARLO SIMULATIONS
Running 10,000 simulations per game using quantile-based triangular distributions...

🎲 Game 1/8: Atlanta Hawks @ Philadelphia 76ers... ✅ Atlanta Hawks (72.0%)
🎲 Game 2/8: Indiana Pacers @ Washington Wizards... ✅ Indiana Pacers (80.1%)
🎲 Game 3/8: Detroit Pistons @ New York Knicks... ✅ New York Knicks (59.8%)
🎲 Game 4/8: Toronto Raptors @ Chicago Bulls... ✅ Toronto Raptors (50.5%)
🎲 Game 5/8: Phoenix Suns @ San Antonio Spurs... ✅ Phoenix Suns (50.9%)
🎲 Game 6/8: Boston Celtics @ Golden State Warriors... ✅ Boston Celtics (61.9%)
🎲 Game 7/8: Orlando Magic @ Sacramento Kings... ✅ Orlando Magic (66.0%)
🎲 Game 8/8: Denver Nuggets @ Los Angeles Clippers... ✅ Los Angeles Clippers (61.2%)

✅ Monte Carlo complete: 8 games simulated (80,000 total simulations)


In [52]:
# ============================================================
# SAVE TO SQLITE DATABASE
# ============================================================
print("\n" + "="*80)
print("💾 SAVING TO SQLITE DATABASE")
print("="*80 + "\n")

import sqlite3
from datetime import datetime

# Create/connect to database
db_path = 'predictions.db'
conn = sqlite3.connect(db_path)

# Create predictions table if it doesn't exist
conn.execute('''
CREATE TABLE IF NOT EXISTS predictions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    prediction_date TEXT,
    game_date TEXT,
    game_num INTEGER,
    away_team TEXT,
    home_team TEXT,
    game_time TEXT,
    ensemble_winner TEXT,
    ensemble_probability REAL,
    ensemble_spread REAL,
    ensemble_confidence TEXT,
    ensemble_confidence_score REAL,
    mc_winner TEXT,
    mc_win_prob REAL,
    mc_median_spread REAL,
    mc_mean_spread REAL,
    mc_std_spread REAL,
    ci_95_low REAL,
    ci_95_high REAL,
    ci_80_low REAL,
    ci_80_high REAL,
    q10 REAL,
    q50 REAL,
    q90 REAL,
    model_confidence TEXT,
    methods_agree INTEGER,
    created_at TEXT
)
''')

# Merge ensemble and MC results
prediction_date = datetime.now().strftime('%Y-%m-%d')
game_date = '2026-02-19'
created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Delete existing predictions for this game date to avoid duplicates
conn.execute('DELETE FROM predictions WHERE game_date = ?', (game_date,))
print(f"🗑️  Cleared existing predictions for {game_date}")

for idx, row in predictions_df.iterrows():
    mc_row = mc_df[mc_df['game_num'] == row['game_num']].iloc[0] if len(mc_df[mc_df['game_num'] == row['game_num']]) > 0 else None
    
    if mc_row is not None:
        methods_agree = 1 if row['winner'] == mc_row['mc_winner'] else 0
        
        conn.execute('''
        INSERT INTO predictions (
            prediction_date, game_date, game_num, away_team, home_team, game_time,
            ensemble_winner, ensemble_probability, ensemble_spread, ensemble_confidence, ensemble_confidence_score,
            mc_winner, mc_win_prob, mc_median_spread, mc_mean_spread, mc_std_spread,
            ci_95_low, ci_95_high, ci_80_low, ci_80_high,
            q10, q50, q90, model_confidence, methods_agree, created_at
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            prediction_date, game_date, row['game_num'], row['away_team'], row['home_team'], row['time'],
            row['winner'], row['probability'], row['spread'], row['confidence'], row['confidence_score'],
            mc_row['mc_winner'], mc_row['mc_win_prob'], mc_row['mc_median_spread'], 
            mc_row['mc_mean_spread'], mc_row['mc_std_spread'],
            mc_row['ci_95_low'], mc_row['ci_95_high'], mc_row['ci_80_low'], mc_row['ci_80_high'],
            mc_row['q10'], mc_row['q50'], mc_row['q90'], mc_row['model_confidence'],
            methods_agree, created_at
        ))

conn.commit()

# Verify save
result = conn.execute(f"SELECT COUNT(*) FROM predictions WHERE prediction_date = '{prediction_date}'").fetchone()
print(f"✅ Saved {result[0]} predictions to {db_path}")
print(f"   Prediction date: {prediction_date}")
print(f"   Game date: {game_date}")
print(f"   Created at: {created_at}\n")

conn.close()


💾 SAVING TO SQLITE DATABASE

🗑️  Cleared existing predictions for 2026-02-19
✅ Saved 8 predictions to predictions.db
   Prediction date: 2026-02-16
   Game date: 2026-02-19
   Created at: 2026-02-16 22:10:26



In [53]:
# ============================================================
# ENSEMBLE vs MONTE CARLO COMPARISON
# ============================================================
print("="*80)
print("⚖️  ENSEMBLE vs MONTE CARLO COMPARISON")
print("="*80 + "\n")

# Create comparison dataframe
comparison_data = []

for idx, ens_row in predictions_df.iterrows():
    mc_row = mc_df[mc_df['game_num'] == ens_row['game_num']].iloc[0] if len(mc_df[mc_df['game_num'] == ens_row['game_num']]) > 0 else None
    
    if mc_row is not None:
        agree = '✅' if ens_row['winner'] == mc_row['mc_winner'] else '❌'
        
        comparison_data.append({
            'Game': f"{ens_row['game_num']}. {ens_row['away_team'][:12]} @ {ens_row['home_team'][:12]}",
            'Ens Winner': ens_row['winner'][:15],
            'Ens Prob': f"{ens_row['probability']:.1%}",
            'MC Winner': mc_row['mc_winner'][:15],
            'MC Prob': f"{mc_row['mc_win_prob']:.1%}",
            'Agree': agree,
            'MC 95% CI': f"[{mc_row['ci_95_low']:+.1f}, {mc_row['ci_95_high']:+.1f}]",
        })

comparison_df = pd.DataFrame(comparison_data)

print(comparison_df.to_string(index=False))

# Summary statistics
agree_count = (comparison_df['Agree'] == '✅').sum()
disagree_count = (comparison_df['Agree'] == '❌').sum()

print(f"\n{'='*80}")
print(f"📊 AGREEMENT SUMMARY:")
print(f"   Methods agree:    {agree_count}/8 games ({agree_count/8:.1%})")
print(f"   Methods disagree: {disagree_count}/8 games ({disagree_count/8:.1%})")

if disagree_count > 0:
    print(f"\n⚠️  DIVERGENT PREDICTIONS:")
    for idx, row in comparison_df[comparison_df['Agree'] == '❌'].iterrows():
        print(f"   {row['Game']}")
        print(f"      Ensemble: {row['Ens Winner']:15} ({row['Ens Prob']})")
        print(f"      MC Sims:  {row['MC Winner']:15} ({row['MC Prob']})")
        print()

print("="*80)
print("✅ ANALYSIS COMPLETE - Results saved to predictions.db")
print("="*80 + "\n")

⚖️  ENSEMBLE vs MONTE CARLO COMPARISON

                          Game      Ens Winner Ens Prob       MC Winner MC Prob Agree      MC 95% CI
1. Atlanta Hawk @ Philadelphia Philadelphia 76    53.3%   Atlanta Hawks   72.0%     ❌  [-18.7, +9.2]
2. Indiana Pace @ Washington W  Indiana Pacers    58.6%  Indiana Pacers   80.1%     ✅  [-21.2, +8.8]
3. Detroit Pist @ New York Kni New York Knicks    66.2% New York Knicks   59.8%     ✅ [-11.4, +14.1]
4. Toronto Rapt @ Chicago Bull   Chicago Bulls    56.2% Toronto Raptors   50.5%     ❌ [-15.3, +14.6]
5. Phoenix Suns @ San Antonio     Phoenix Suns    58.7%    Phoenix Suns   50.9%     ✅ [-16.4, +14.8]
6. Boston Celti @ Golden State  Boston Celtics    50.4%  Boston Celtics   61.9%     ✅ [-15.5, +11.3]
7. Orlando Magi @ Sacramento K Sacramento King    52.2%   Orlando Magic   66.0%     ❌ [-15.7, +10.1]
8. Denver Nugge @ Los Angeles   Denver Nuggets    56.4% Los Angeles Cli   61.2%     ❌ [-12.1, +14.8]

📊 AGREEMENT SUMMARY:
   Methods agree:    4/8 game